In [9]:
import anndata as ad
import os
import requests
import scprep
import tempfile
import time
from utils import *

In [2]:
URL = "https://figshare.com/ndownloader/files/36509385"

def load_mouse_brain_atlas(test=False):
    """Download Allen Brain (Taisc et al.,2016) data from Figshare.

    Further information how we generated the assumed truth and the reference
    to the dataset is available at:
    https://figshare.com/articles/dataset/allen_brain_h5ad/20338089
    """
    import scanpy as sc

    if test:
        # load full data first, cached if available
        adata = load_mouse_brain_atlas(test=False)

        # Subsample data to random cell from each cell type
        msk = adata.obs_names.isin(adata.obs.groupby("label").head(20).index)
        adata = adata[msk, :]

        # Basic filtering
        filter_genes_cells(adata)

        # Keep only 500 genes
        adata = adata[:, 30000:30500].copy()

        # remove missing labels for tests
        labels = adata.obs["label"].cat.categories
        source_mask = ~adata.uns["ccc_target"]["target"].isin(labels)
        target_mask = ~adata.uns["ccc_target"]["target"].isin(labels)
        msk = np.logical_not(source_mask | target_mask)
        adata.uns["ccc_target"] = adata.uns["ccc_target"][msk]

        return adata

    else:
        with tempfile.TemporaryDirectory() as tempdir:
            filepath = os.path.join(tempdir, "mouse_brain_atlas.h5ad")
            scprep.io.download.download_url(URL, filepath)
            adata = sc.read(filepath)

            # Ensure there are no cells or genes with 0 counts
            filter_genes_cells(adata)

        return adata

In [12]:
from typing import Union

import anndata
import collections
import numpy as np
import pandas as pd
import pathlib
import scipy.sparse
import scprep.run

# Helper function to obtain and convert a Ligand-Receptor resource
ligand_receptor_resource = scprep.run.RFunction(
    args="target_organism",
    body="""
        HUMAN <- 9606
        if (target_organism != HUMAN) {
            op_resource <- liana::generate_homologs(
                op_resource = liana::select_resource('Consensus')[[1]],
                target_organism = target_organism
            )
        } else {
            op_resource <- liana::select_resource('Consensus')[[1]]
        }

        dplyr::mutate(
            op_resource,
            ligand_genesymbol = source_genesymbol,
            receptor_genesymbol = target_genesymbol
        )
    """,
)


def map_gene_symbols(adata, map_filename: Union[str, pathlib.Path]):
    """Maps gene symbols from aliases to standard symbols

    Genes that map many-to-one are summed.
    Genes that map one-to-many are duplicated.

    Parameters
    ----------
    adata : anndata.AnnData
    map_filename : PathLike
        Path to csv containing gene symbol map with two columns, `gene` and `alias`

    Returns
    -------
    adata : anndata.AnnData
    """
    map_df = pd.read_csv(map_filename)
    var = adata.var.rename_axis("alias", axis=0)[[]]
    gene_match_idx = np.isin(var.index, map_df["gene"])
    var_gene_match, var = var.loc[gene_match_idx].copy(), var.loc[~gene_match_idx]
    alias_match_idx = np.isin(var.index, map_df["alias"])
    var_alias_match, var_no_map = (
        var.loc[alias_match_idx].copy(),
        var.loc[~alias_match_idx].copy(),
    )

    # fill 'gene' column
    var_alias_match = var_alias_match.reset_index().merge(
        map_df, on="alias", how="left"
    )
    var_gene_match["gene"] = var_gene_match.index
    var_no_map["gene"] = var_no_map.index

    var_dealiased = pd.concat(
        [var_gene_match.reset_index(), var_no_map.reset_index(), var_alias_match]
    )
    duplicate_idx = var_dealiased["gene"].duplicated(keep=False)
    var_dealiased_many_to_one, var_dealiased_one_to_any = (
        var_dealiased.loc[duplicate_idx],
        var_dealiased.loc[~duplicate_idx],
    )

    adata_one_to_any = adata[:, var_dealiased_one_to_any["alias"]]
    adata_one_to_any.var.index = var_dealiased_one_to_any["gene"]

    many_to_one_genes = var_dealiased_many_to_one["gene"].unique()
    many_to_one_X = []
    many_to_one_layers = collections.defaultdict(list)
    for gene in var_dealiased_many_to_one["gene"].unique():
        gene_aliases = var_dealiased_many_to_one.loc[
            var_dealiased_many_to_one["gene"] == gene, "alias"
        ]
        adata_gene = adata[:, gene_aliases]
        many_to_one_X.append(scipy.sparse.coo_matrix(adata_gene.X.sum(axis=1)))
        for layer_name, layer in adata_gene.layers.items():
            many_to_one_layers[layer_name].append(
                scipy.sparse.coo_matrix(adata_gene.X.sum(axis=1))
            )

    return anndata.AnnData(
        X=scipy.sparse.hstack([adata_one_to_any.X] + many_to_one_X).tocsr(),
        obs=adata.obs,
        var=pd.DataFrame(
            index=np.concatenate([adata_one_to_any.var.index, many_to_one_genes])
        ),
        layers={
            layer_name: scipy.sparse.hstack(
                [adata_one_to_any.layers[layer_name]] + many_to_one_layers[layer_name]
            ).tocsr()
            for layer_name in adata.layers
        },
        uns=adata.uns,
        obsm=adata.obsm,
    )


# Join predictions to target
def join_truth_and_pred(adata):
    merge_keys = list(adata.uns["merge_keys"])
    gt = adata.uns["ccc_target"].merge(adata.uns["ccc_pred"], on=merge_keys, how="left")

    gt.loc[gt["response"].isna(), "response"] = 0
    gt.loc[gt["score"].isna(), "score"] = np.nanmin(gt["score"]) - np.finfo(float).eps

    return gt


def aggregate_method_scores(adata, how):
    merge_keys = list(adata.uns["merge_keys"])
    return (
        adata.uns["ccc_pred"]
        .groupby(merge_keys)
        .agg(score=("score", how))
        .reset_index()
    )

In [4]:
adata = load_mouse_brain_atlas(test=False)
adata

AnnData object with n_obs × n_vars = 14249 × 34617
    obs: 'label', 'sizeFactor', 'subclass', 'n_counts'
    var: 'n_cells'
    uns: '_from_cache', 'age_days', 'brain_hemisphere', 'brain_region', 'brain_subregion', 'ccc_target', 'class', 'cluster', 'cluster_correlation', 'complexity_cg', 'confusion_score', 'core_intermediate_call', 'donor', 'driver_lines', 'eye_condition', 'facs_container', 'facs_date', 'facs_sort_criteria', 'genes_detected_cpm_criterion', 'genes_detected_fpkm_criterion', 'genotype', 'gfp_cpm', 'injection_exclusion_criterion', 'injection_label_direction', 'injection_material', 'injection_primary', 'injection_secondary', 'injection_tract', 'library_prep_avg_size_bp', 'library_prep_set', 'nCount_RNA', 'nFeature_RNA', 'organism', 'orig.ident', 'percent_aligned_reads_total', 'percent_ecoli_reads', 'percent_exon_reads', 'percent_intergenic_reads', 'percent_intron_reads', 'percent_mt_exon_reads', 'percent_reads_unique', 'percent_rrna_reads', 'percent_synth_reads', 'reporter

In [5]:
adata.uns["merge_keys"] = ["source", "target"]
adata

AnnData object with n_obs × n_vars = 14249 × 34617
    obs: 'label', 'sizeFactor', 'subclass', 'n_counts'
    var: 'n_cells'
    uns: '_from_cache', 'age_days', 'brain_hemisphere', 'brain_region', 'brain_subregion', 'ccc_target', 'class', 'cluster', 'cluster_correlation', 'complexity_cg', 'confusion_score', 'core_intermediate_call', 'donor', 'driver_lines', 'eye_condition', 'facs_container', 'facs_date', 'facs_sort_criteria', 'genes_detected_cpm_criterion', 'genes_detected_fpkm_criterion', 'genotype', 'gfp_cpm', 'injection_exclusion_criterion', 'injection_label_direction', 'injection_material', 'injection_primary', 'injection_secondary', 'injection_tract', 'library_prep_avg_size_bp', 'library_prep_set', 'nCount_RNA', 'nFeature_RNA', 'organism', 'orig.ident', 'percent_aligned_reads_total', 'percent_ecoli_reads', 'percent_exon_reads', 'percent_intergenic_reads', 'percent_intron_reads', 'percent_mt_exon_reads', 'percent_reads_unique', 'percent_rrna_reads', 'percent_synth_reads', 'reporter

In [10]:
adata.uns["merge_keys"]

['source', 'target']

In [7]:
adata.uns["target_organism"]

10090

In [8]:
adata.uns["ligand_receptor_resource"] = ligand_receptor_resource(
    adata.uns["target_organism"]
)

adata

/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/local/nvidia/lib:/usr/local/nvidia/lib64", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/local/nvidia/lib:/usr/local/nvidia/lib64"
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "PWD" redefined by R and overriding existing variable. Current: "/app", R: "/app/Machine-learning-development-environment-for-single-cell-sequencing-data-analyses/api/test"
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/local/nvidia/lib:/usr/local/nvidia/lib64", R: "/usr/lib


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

--- Logging error ---
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/logging/__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/logging/__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/logging/__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/logging/__init__.py", line 377, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.p

|======================================================|100% ~0 s remaining     

AnnData object with n_obs × n_vars = 14249 × 34617
    obs: 'label', 'sizeFactor', 'subclass', 'n_counts'
    var: 'n_cells'
    uns: '_from_cache', 'age_days', 'brain_hemisphere', 'brain_region', 'brain_subregion', 'ccc_target', 'class', 'cluster', 'cluster_correlation', 'complexity_cg', 'confusion_score', 'core_intermediate_call', 'donor', 'driver_lines', 'eye_condition', 'facs_container', 'facs_date', 'facs_sort_criteria', 'genes_detected_cpm_criterion', 'genes_detected_fpkm_criterion', 'genotype', 'gfp_cpm', 'injection_exclusion_criterion', 'injection_label_direction', 'injection_material', 'injection_primary', 'injection_secondary', 'injection_tract', 'library_prep_avg_size_bp', 'library_prep_set', 'nCount_RNA', 'nFeature_RNA', 'organism', 'orig.ident', 'percent_aligned_reads_total', 'percent_ecoli_reads', 'percent_exon_reads', 'percent_intergenic_reads', 'percent_intron_reads', 'percent_mt_exon_reads', 'percent_reads_unique', 'percent_rrna_reads', 'percent_synth_reads', 'reporter

In [11]:
adata.uns["ligand_receptor_resource"]

,source_genesymbol,target_genesymbol,source,target,category_intercell_source,database_intercell_source,category_intercell_target,database_intercell_target,sources,references,ligand_genesymbol,receptor_genesymbol
1,Dll1,Notch1,O00548,P46531,cell_surface_ligand,connectomeDB2020;CellChatDB;CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,Baccin2019;CellCall;CellChatDB;CellPhoneDB;Cel...,Baccin2019:1006133;Baccin2019:98194281;CellCha...,Dll1,Notch1
2,Dll1,Notch2,O00548,Q04721,cell_surface_ligand,connectomeDB2020;CellChatDB;CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,Baccin2019;BioGRID;CellCall;CellChatDB;CellPho...,Baccin2019:10958687;BioGRID:9244302;CellChatDB...,Dll1,Notch2
3,Dll1,Notch4,O00548,Q99466,cell_surface_ligand,connectomeDB2020;CellChatDB;CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,CellCall;CellChatDB;CellPhoneDB;CellPhoneDB_Ce...,CellChatDB:22353464;CellPhoneDB:22353464;CellT...,Dll1,Notch4
4,Dll1,Notch3,O00548,Q9UM47,cell_surface_ligand,connectomeDB2020;CellChatDB;CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,Baccin2019;CancerCellMap;CellCall;CellChatDB;C...,Baccin2019:11006133;Baccin2019:12482954;Cancer...,Dll1,Notch3
5,Nrg2,Erbb2_Erbb3,O14511,COMPLEX:P04626_P21860,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellChatDB;CellTalkDB,CellChatDB;Cellinker;Guide2Pharma_Cellinker;HP...,Cellinker:9168114;Cellinker:9168115,Nrg2,Erbb2_Erbb3
...,...,...,...,...,...,...,...,...,...,...,...,...
3994,Serpina1a,Lrp1,P01009,Q07954,,,,,Baccin2019;CellTalkDB;Cellinker;Fantom5_LRdb;H...,Baccin2019:862645615131125;CellTalkDB:8626456;...,Serpina1a,Lrp1
3995,Serpina1b,Lrp1,P01009,Q07954,,,,,Baccin2019;CellTalkDB;Cellinker;Fantom5_LRdb;H...,Baccin2019:862645615131125;CellTalkDB:8626456;...,Serpina1b,Lrp1
3996,Serpina1c,Lrp1,P01009,Q07954,,,,,Baccin2019;CellTalkDB;Cellinker;Fantom5_LRdb;H...,Baccin2019:862645615131125;CellTalkDB:8626456;...,Serpina1c,Lrp1
3997,Serpina1d,Lrp1,P01009,Q07954,,,,,Baccin2019;CellTalkDB;Cellinker;Fantom5_LRdb;H...,Baccin2019:862645615131125;CellTalkDB:8626456;...,Serpina1d,Lrp1


In [12]:
adata.write_h5ad('../../../oscb/mouse_brain_atlas.h5ad')

In [13]:
adata.uns['ccc_target']

,source,target,response
0,CR,Astro,0
1,Endo,Astro,1
2,L2.3.IT,Astro,0
3,L4,Astro,0
4,L5.IT,Astro,0
...,...,...,...
501,Serpinf1,VLMC,0
502,SMC,VLMC,0
503,Sncg,VLMC,0
504,Sst,VLMC,0


# Ligand-Target

In [13]:
URL = "https://figshare.com/ndownloader/files/37593188"
tempdir = '../../../oscb/uploads/'

def load_tnbc_data(test=False):
    """Download TNBC data (Wu et al., 2021) from Figshare.

    Further information how we generated the assumed truth and the reference
    to the dataset is available at:
    https://figshare.com/articles/dataset/TNBC_Data_from_Wu_et_al_2021/20338536

    """
    import scanpy as sc

    if test:
        # load full data first, cached if available
        adata = load_tnbc_data(test=False)

        # Keep only relevant response ligands
        lr = list(set(adata.uns["ccc_target"].ligand))
        # add corresponding receptors
        lr += [
            "IFNLR1",
            "CD70",
            "NGFR",
            "CD53",
            "IL20RA",
            "IL20RB",
            "IL2RG",
            "SIGLEC6",
            "EGFR",
            "ITGB1",
            "ITGB3",
            "ACVRL1",
            "CXCR4",
            "SMAD3",
            "CAV1",
            "IL21R",
            "IL4R",
            "IL2RG",
            "IL13RA1",
            "IL13RA2",
        ]
        adata = adata[:, adata.var.index.isin(lr)].copy()

        # Keep only cells with nonzero relevant ligands/receptors
        filter_genes_cells(adata)

        # Subsample data to random cell from each cell type
        adata = subsample_even(adata, n_obs=500, even_obs="label")

        # Add noise to ensure no genes are empty -- filtering genes could remove ligands
        empty_gene_idx = np.argwhere(adata.X.sum(axis=0).A.flatten() == 0).flatten()
        random_cell_idx = np.random.choice(adata.shape[0], len(empty_gene_idx))
        adata.X += scipy.sparse.coo_matrix(
            (
                np.ones(len(empty_gene_idx), dtype=adata.X.dtype),
                (random_cell_idx, empty_gene_idx),
            ),
            shape=adata.shape,
            dtype=adata.X.dtype,
        )
        adata.X = adata.X.tocsr()
        # update layers["counts"] with the new counts
        adata.layers["counts"] = adata.X
    else:
        filepath = os.path.join(tempdir, "brca_tnbc.h5ad")
        scprep.io.download.download_url(URL, filepath)
        adata = sc.read(filepath)

        # Ensure there are no cells or genes with 0 counts
        filter_genes_cells(adata)

    # Remove unneeded cols
    adata.uns["ccc_target"] = adata.uns["ccc_target"][["ligand", "target", "response"]]

    return adata

In [14]:
adata = load_tnbc_data(test=False)
adata

AnnData object with n_obs × n_vars = 42512 × 28200
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'ident', 'label', 'n_counts'
    var: 'n_cells'
    uns: 'ccc_target', 'target_organism', 'var_names_all'

In [16]:
import pathlib

adata = map_gene_symbols(
    adata, "tnbc_wu2021_gene_symbols.csv"
)

/tmp/ipykernel_192/3025814964.py:77: ImplicitModificationWarning: Trying to modify index of attribute `.var` of view, initializing view as actual.
  adata_one_to_any.var.index = var_dealiased_one_to_any["gene"]
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [17]:
adata.uns["merge_keys"] = ["ligand", "target"]

In [18]:
adata.uns["ligand_receptor_resource"] = ligand_receptor_resource(
        adata.uns["target_organism"]
    )

/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/local/nvidia/lib:/usr/local/nvidia/lib64", R: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/local/nvidia/lib:/usr/local/nvidia/lib64"
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "PWD" redefined by R and overriding existing variable. Current: "/app", R: "/app/Machine-learning-development-environment-for-single-cell-sequencing-data-analyses/api/test"
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/rpy2/rinterface/__init__.py:1211: UserWarning: Environment variable "LD_LIBRARY_PATH" redefined by R and overriding existing variable. Current: "/usr/lib/R/lib:/usr/lib/x86_64-linux-gnu:/usr/lib/jvm/default-java/lib/server:/usr/local/nvidia/lib:/usr/local/nvidia/lib64", R: "/usr/lib


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

In [20]:
adata.uns['ligand_receptor_resource']

,source,target,source_genesymbol,target_genesymbol,category_intercell_source,database_intercell_source,category_intercell_target,database_intercell_target,sources,references,ligand_genesymbol,receptor_genesymbol
1,O00182,P08575,LGALS9,PTPRC,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellChatDB;CellTalkDB,CellChatDB;Cellinker,CellChatDB:30120235;Cellinker:30120235,LGALS9,PTPRC
2,O00182,P08581,LGALS9,MET,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,CellPhoneDB;InnateDB-All_CellPhoneDB,,LGALS9,MET
3,O00182,P16070,LGALS9,CD44,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;ICELLNET;CellChat...,CellChatDB;CellPhoneDB;CellTalkDB;Cellinker;In...,CellChatDB:25065622;CellTalkDB:32103204;Cellin...,LGALS9,CD44
4,O00182,Q07954,LGALS9,LRP1,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;CellChatDB;CellTa...,CellPhoneDB;InnateDB-All_CellPhoneDB,,LGALS9,LRP1
5,O00182,Q08722,LGALS9,CD47,cell_surface_ligand,CellPhoneDB,receptor,connectomeDB2020;CellPhoneDB;CellChatDB;CellTa...,CellPhoneDB;CellTalkDB;InnateDB-All_CellPhoneDB,CellTalkDB:32103204,LGALS9,CD47
...,...,...,...,...,...,...,...,...,...,...,...,...
4697,P12643,P61160,BMP2,ACTR2,,,,,SignaLink3,SignaLink3:16446785;SignaLink3:23331499,BMP2,ACTR2
4698,O95972,P61160,BMP15,ACTR2,,,,,SignaLink3,SignaLink3:16446785;SignaLink3:23331499,BMP15,ACTR2
4699,P09603,Q99062,CSF1,CSF3R,,,,,SIGNOR;SignaLink3,SIGNOR:16492764;SignaLink3:16492764;SignaLink3...,CSF1,CSF3R
4700,Q9NZH8,P17181,IL36G,IFNAR1,,,,,SignaLink3,SignaLink3:17208301;SignaLink3:23331499,IL36G,IFNAR1


In [26]:
adata.write_h5ad('../../../oscb/brca_tnbc.h5ad')

In [23]:
adata.X.todense()

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.]], dtype=float32)

In [24]:
adata.uns["merge_keys"]

['ligand', 'target']

In [27]:
adata.uns["ccc_target"]

,ligand,target,response
0,BDNF,B.cells.Memory,0
1,BMP2,B.cells.Memory,0
2,BMP4,B.cells.Memory,0
3,BMP6,B.cells.Memory,0
4,CXCL12,B.cells.Memory,0
...,...,...,...
807,OSM,T.cells.CD8,0
808,TGFB1,T.cells.CD8,0
809,TGFB3,T.cells.CD8,0
810,VEGFA,T.cells.CD8,0
